In [ ]:
#install clip
!pip install ftfy regex tqdm
!pip install openai_clip

In [ ]:
#necessary imports
import torch
import torchvision
import torch.nn as nn
import clip
from torch.nn import functional as F
from tqdm import tqdm
from collections import OrderedDict

We will now create functions to correctly get our data and split it into base and novel classes.

In [ ]:
def get_data(data_dir="./data", transform=None):
    """Load Flowers102 train, validation and test sets.
    Args:
        data_dir (str): Directory where the dataset will be stored.
        transform (torch.Compose)
    Returns:
        tuple: A tuple containing the train, validation, and test sets.
    """
    train = torchvision.datasets.Flowers102(root=data_dir, split="train", download=True, transform=transform)
    val = torchvision.datasets.Flowers102(root=data_dir, split="val", download=True, transform=transform)
    test = torchvision.datasets.Flowers102(root=data_dir, split="test", download=True, transform=transform)
    return train, val, test

In [ ]:
def base_novel_categories(dataset):
    """Splits classes into base and novel categories.
    Args:
        dataset: The dataset containing the classes.
    Returns:
        A tuple containing two lists: the base classes and the novel classes.
    """
    all_classes = set(dataset._labels)
    num_classes = len(all_classes)
    base_classes = list(range(num_classes))[:num_classes//2]
    novel_classes = list(range(num_classes))[num_classes//2:]
    return base_classes, novel_classes

In [ ]:
_, _, tmp_test = get_data()
base_classes, novel_classes = base_novel_categories(tmp_test)
CLASS_NAMES = ["pink primrose", "hard-leaved pocket orchid", "canterbury bells", "sweet pea",
                "english marigold", "tiger lily", "moon orchid", "bird of paradise", "monkshood",
                "globe thistle", "snapdragon", "colt's foot", "king protea", "spear thistle",
                "yellow iris", "globe-flower", "purple coneflower", "   ", "balloon flower",
                "giant white arum lily", "fire lily", "pincushion flower", "fritillary", "red ginger",
                "grape hyacinth", "corn poppy", "prince of wales feathers", "stemless gentian", "artichoke",
                "sweet william", "carnation", "garden phlox", "love in the mist", "mexican aster",
                "alpine sea holly", "ruby-lipped cattleya", "cape flower", "great masterwort", "siam tulip",
                "lenten rose", "barbeton daisy", "daffodil", "sword lily", "poinsettia", "bolero deep blue",
                "wallflower", "marigold", "buttercup", "oxeye daisy", "common dandelion", "petunia", "wild pansy",
                "primula", "sunflower", "pelargonium", "bishop of llandaff", "gaura", "geranium", "orange dahlia",
                "pink-yellow dahlia", "cautleya spicata", "japanese anemone", "black-eyed susan", "silverbush",
                "californian poppy", "osteospermum", "spring crocus", "bearded iris", "windflower", "tree poppy",
                "gazania", "azalea", "water lily", "rose", "thorn apple", "morning glory", "passion flower", "lotus",
                "toad lily", "anthurium", "frangipani", "clematis", "hibiscus", "columbine", "desert-rose",
                "tree mallow", "magnolia", "cyclamen", "watercress", "canna lily", "hippeastrum", "bee balm",
                "ball moss", "foxglove", "bougainvillea", "camellia", "mallow", "mexican petunia", "bromelia",
                "blanket flower", "trumpet creeper", "blackberry lily"]
print("Base Class Names:", [(i, CLASS_NAMES[i]) for i in base_classes])
print("Novel Class Names:", [(i, CLASS_NAMES[i]) for i in novel_classes])

Let's now split the dataset.

In [ ]:
def split_data(dataset, base_classes):
    """Splits the dataset into base and novel category subsets.
    Args:
        dataset: The dataset containing the samples.
        base_classes: The list of base class indices.
    Returns:
        A tuple containing the base and novel category datasets.
    """
    base_categories_samples = []
    novel_categories_samples = []
    base_set = set(base_classes)
    for sample_id, label in enumerate(dataset._labels):
        if label in base_set:
            base_categories_samples.append(sample_id)
        else:
            novel_categories_samples.append(sample_id)
    base_dataset = torch.utils.data.Subset(dataset, base_categories_samples)
    novel_dataset = torch.utils.data.Subset(dataset, novel_categories_samples)
    return base_dataset, novel_dataset

In [ ]:
def create_remapped_dataset(dataset, selected_classes):
    """Create a dataset subset with remapped labels.
    Args:
        dataset: Original dataset
        selected_classes: List of class indices to include
    Returns:
        subset dataset with labels remapped to [0, len(selected_classes)-1]
    """
    # Create mapping from original labels to new labels
    label_map = {old_label: new_label for new_label, old_label in enumerate(selected_classes)}
    selected_set = set(selected_classes)

    # Find samples and create new labels
    selected_samples = []
    new_labels = []

    for sample_id, label in enumerate(dataset._labels):
        if label in selected_set:
            selected_samples.append(sample_id)
            new_labels.append(label_map[label])

    # Create subset
    subset = torch.utils.data.Subset(dataset, selected_samples)

    # Add remapped labels to subset
    subset.remapped_labels = new_labels

    return subset

class RemappedDataset(torch.utils.data.Dataset):
    """Wrapper dataset that returns remapped labels"""
    def __init__(self, subset_dataset):
        self.dataset = subset_dataset.dataset
        self.indices = subset_dataset.indices
        self.labels = subset_dataset.remapped_labels

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        # Get original sample
        original_idx = self.indices[idx]
        image, _ = self.dataset[original_idx]  # Ignore original label

        # Return with remapped label
        return image, self.labels[idx]

Let's now load a pretrained CLIP model.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

# Load the CLIP model and preprocessing transform
model, preprocess = clip.load("ViT-B/16", device=device)

# Cast the model to float32 if on GPU to prevent dtype errors
if device == "cuda":
    model.float()

model.eval()  # We won't fine-tune CLIP
for param in model.parameters():
    param.requires_grad = False

# Get the image and text encoders
image_encoder = model.visual
text_encoder = model.encode_text  # Used in inference mode only

Let's prepare the train-test-splits.


In [ ]:
# get the three datasets
train_set, val_set, test_set = get_data(transform=preprocess)

# split classes into base and novel
base_classes, novel_classes = base_novel_categories(train_set)

# split the three datasets
train_base, _ = split_data(train_set, base_classes)
val_base, _ = split_data(val_set, base_classes)
test_base, test_novel = split_data(test_set, base_classes)

### Now we will start implementing MoCoOp: Mixture of Prompt Learning


#### Utils

In [ ]:
def l2norm(x, dim=-1, eps=1e-8):
    """
    Applies L2 normalization to the input tensor along the specified dimension.
    Args:
        x (torch.Tensor): Input tensor to normalize.
        dim (int): Dimension along which to compute the norm.
        eps (float): Small value to avoid division by zero.
    Returns:
        torch.Tensor: L2-normalized tensor.
    """
    return x / (x.norm(dim=dim, keepdim=True) + eps)

# 8 groups of hard prompts for Flowers102
HARD_GROUPS = [
    ["a photo of a {}, a type of flower.", "a photo of the {}, a type of flower. "], # flowers
    ["a photo of a {}.", "a photo of the {}."],                                      # generic
    ["a close-up photo of a {}.", "a macro photo of a {}."],                         # proximity
    ["a good photo of a {}.", "a good quality photo of a {}."]                       # good quality
]


@torch.no_grad()
def encode_texts(clip_model, texts):
    """
    Encodes a list of text prompts into CLIP text feature embeddings.
    Args:
        clip_model: The loaded CLIP model.
        texts: List of text prompts to encode.
    Returns:
        torch.Tensor: L2-normalized text feature embeddings of shape [B, 512],
                      where B is the number of input texts and 512 is the CLIP text embedding dimension.
    """
    device = next(clip_model.parameters()).device
    tokenized = clip.tokenize(texts).to(device)  # [B, 77] - B is batch size, 77 is CLIP's max context length
    features = clip_model.encode_text(tokenized).float()  # [B, 512] - 512 is CLIP's text embedding dim
    return l2norm(features)

@torch.no_grad()
def build_hard_features(clip_model, class_names, hard_groups):
    """
    Computes CLIP text feature embeddings for groups of hard prompts and all class names.
    Args:
        clip_model: The loaded CLIP model.
        class_names (list of str): List of class names.
        hard_groups (list of list of str): List of prompt groups, each group is a list of prompt templates containing '{}' for class name.

    Returns:
        hard_group_avg (torch.Tensor): [G, D] tensor, where G = number of groups, D = embedding dim.
            Each row is the average feature for a group, averaged over all classes. Will be used for router supervision.
        hard_group_class (torch.Tensor): [G, C, D] tensor, where C = number of classes.
            Each row is the average feature for a group and class, averaged over prompts in the group.
            Will be used for text-level supervision.

    Details:
        - For each group of prompts, and for each class name:
            - Formats each prompt with the class name.
            - Encodes all prompts for the class using CLIP.
            - Averages the features for the group/class.
        - Normalizes all features with L2 norm.
        - Returns both per-group averages and per-group/class averages.
    """
    group_avg = []
    group_class = []
    for group in hard_groups:
        per_class_features = []
        for class_name in class_names:
            texts = [prompt.format(class_name) for prompt in group] # insert class name at placeholder {} in each prompt
            features = encode_texts(clip_model, texts).mean(0,keepdim=True)  # [1, D] - average across the 2 prompt embeddings in the group
            per_class_features.append(features)
        per_class_features = torch.cat(per_class_features, dim=0) # [C, D]
        per_class_features = l2norm(per_class_features)
        group_class.append(per_class_features)
        group_avg.append(per_class_features.mean(0, keepdim=True))
    hard_group_avg = l2norm(torch.cat(group_avg, dim=0)) # [G, D]
    hard_group_class = l2norm(torch.stack(group_class, dim=0)) # [G, C, D]
    return hard_group_avg, hard_group_class

#### Router

In [ ]:
class Router(nn.Module):
    """
    Given image features, outputs expert logits.
    Routes inputs towards experts in mixture-of-experts models according to image features,
    using either a linear layer or a two-layer MLP with ReLU and dropout.
    """
    def __init__(self, input_dim, G, hidden_dim=0, bias=True):
        """
        Initialize the Router.

        Args:
            input_dim (int): Dimension of input image features.
            G (int): Number of experts (output dimension).
            hidden_dim (int, optional): If > 0, adds a hidden layer with ReLU and dropout.
            bias (bool, optional): Whether to use bias in linear layers.
        """
        super().__init__()
        if hidden_dim > 0:
            self.net = nn.Sequential(
                nn.Linear(input_dim, hidden_dim, bias=bias),
                nn.ReLU(inplace=True),
                nn.Dropout(p=0.5),
                nn.Linear(hidden_dim, G, bias=bias)
            )
        else:
            self.net = nn.Linear(input_dim, G, bias=bias)

    def forward(self, x):
        """
        Forward pass to compute expert logits.
        Args:
            x (torch.Tensor): Input image features of shape [B, input_dim].
        Returns:
            torch.Tensor: Logits of shape [B, G], where B is batch size and G is number of experts.
        """
        return self.net(x)

#### Text Encoder(we cannot use CLIP's standard text encoder because we learn already embedded prompts, same as in CoOp)

In [ ]:
class TextEncoder(nn.Module):
    """
    Encodes prompts that are already embedded, allowing for learnable prompt contexts.
    It uses CLIP's transformer and projection layers, but does not use CLIP's standard tokenizer pipeline.

    Attributes:
        clip_model: The frozen CLIP model.
        token_prefix: Embedded start-of-text token for each class.
        token_suffix: Embedded suffix tokens for each class.
        tokenized_templates: Tokenized templates for each class.
    """
    def __init__(self, clip_model, token_prefix, token_suffix, tokenized_templates):
        """
        Initializes the TextEncoder.

        Args:
            clip_model: The CLIP model.
            token_prefix (torch.Tensor): Embedded SOT token for each class, shape [C, 1, D].
            token_suffix (torch.Tensor): Embedded suffix tokens for each class, shape [C, S, D].
            tokenized_templates (torch.Tensor): Tokenized templates for each class, shape [C, 77].
        """
        super().__init__()
        self.clip_model = clip_model
        # freeze CLIP
        for p in self.clip_model.parameters():
            p.requires_grad = False #TODO: Review these comments
        self.token_prefix = token_prefix              # [C, 1, D] where C is the number of classes and D is the text embedding dim (512 for CLIP)
        self.token_suffix = token_suffix              # [C, S, D] where S is the number of tokens in the suffices
        self.tokenized_templates = tokenized_templates
        self.num_classes = self.token_prefix.size(0)
        self.eot_idx = self.tokenized_templates.argmax(dim=-1) 


    def forward(self, ctx_g):
        """
        Encodes a prompt context for all classes using CLIP's transformer and projection.

        Args:
            ctx_g (torch.Tensor): Context tokens for the prompt, shape [n_ctx, D].

        Returns:
            torch.Tensor: L2-normalized text features for all classes, shape [C, D].
        """
        context = ctx_g.unsqueeze(0).expand(self.num_classes, -1, -1)  # [C, n_ctx, D]
        x = torch.cat([self.token_prefix, context, self.token_suffix], dim=1)   # [C, seq_len, D]
        x = x + self.clip_model.positional_embedding[:x.size(1)]    # add positional embedding
        x = x.permute(1,0,2)  # [seq_len, C, D]
        x = self.clip_model.transformer(x)    # transformer
        x = x.permute(1,0,2)  # [C, seq_len, D]
        x = self.clip_model.ln_final(x) # layer normalization
        x = x[torch.arange(self.num_classes, device=x.device), self.eot_idx]# extract hidden state at EOT token for each class.
        x = x @ self.clip_model.text_projection# text projection: matrix multiplication
        return l2norm(x)

#### PromptLearner

In [ ]:
class PromptExperts(nn.Module):
    """
    A multi-expert prompt-based classifier using CLIP, designed for few-shot and compositional learning.
    Attributes:
        all_class_names (list): All class names.
        total_class_count (int): Total number of classes.
        active_class_indices (list): Indices of active classes.
        active_class_count (int): Number of active classes.
        active_class_names (list): Names of active classes.
        n_ctx (int): Number of context tokens.
        clip_model (nn.Module): CLIP model (frozen).
        device (torch.device): Device for computation.
        dtype (torch.dtype): Data type for tensors.
        expert_count (int): Number of expert groups.
        top_k (int): Number of top experts to mix.
        tau (float): Temperature for text-level supervision.
        lambda_router (float): Router regularization weight.
        lambda_text (float): Text-level supervision weight.
        ctxs (nn.Parameter): Learnable context parameters for each expert.
        token_prefix (torch.Tensor): Embedding of start-of-text token.
        token_suffix (torch.Tensor): Embedding of suffix tokens.
        tokenized_template_prompts (torch.Tensor): Tokenized template prompts.
        hard_group_avg (torch.Tensor): Hard feature averages for all classes.
        hard_group_class (torch.Tensor): Hard features for all classes.
        hard_group_avg_active (torch.Tensor): Hard feature averages for active classes.
        logit_scale (torch.Tensor): CLIP logit scale.
    """
    def __init__(self, clip_model, all_class_names, active_class_indices=None, n_ctx=16, hard_groups=None,
                top_k=2, tau=0.07, lambda_router=1.0, lambda_text=5.0,
                router_hidden=0):
        """
        Initializes the PromptExperts module.

        Args:
            clip_model (nn.Module): Pretrained CLIP model (frozen).
            all_class_names (list): List of all class names (base + novel).
            active_class_indices (list, optional): Indices of active classes for classification. Defaults to all classes.
            n_ctx (int, optional): Number of context tokens for prompts. Defaults to 16.
            hard_groups (list): List of hard prompt groups, each group is a list of prompt templates containing '{}' for class name.
            top_k (int, optional): Number of top experts to mix per sample. Defaults to 2.
            tau (float, optional): Temperature for text-level supervision. Defaults to 0.07.
            lambda_router (float, optional): Weight for router regularization loss. Defaults to 1.0.
            lambda_text (float, optional): Weight for text-level supervision loss. Defaults to 5.0.
            router_hidden (int, optional): Hidden dimension for router MLP. If 0, uses linear router. Defaults to 0.
        """
        super().__init__()
        assert hard_groups is not None and len(hard_groups) > 0
        self.all_class_names = all_class_names
        self.total_class_count = len(all_class_names)
        if active_class_indices is None:
            active_class_indices = list(range(self.total_class_count))
        self.active_class_indices = active_class_indices
        self.active_class_count = len(active_class_indices)
        self.active_class_names = [all_class_names[i] for i in active_class_indices]
        self.n_ctx = n_ctx
        self.clip_model = clip_model
        # freeze CLIP
        for p in self.clip_model.parameters():
            p.requires_grad = False
        self.device = next(clip_model.parameters()).device
        self.dtype = next(clip_model.parameters()).dtype
        self.expert_count = len(hard_groups)
        self.top_k = top_k
        self.tau = tau
        self.lambda_router = lambda_router
        self.lambda_text = lambda_text

        # soft contexts
        text_embedding_dimension = clip_model.ln_final.weight.shape[0]
        self.ctxs = nn.Parameter(torch.empty(self.expert_count, n_ctx, text_embedding_dimension, dtype=self.dtype)) # [G, n_ctx, D] 

        # templates with placeholders. we use all class names (base + novel)
        prompt_prefix = " ".join(["X"] * n_ctx)
        template_prompts = [f"{prompt_prefix} {name}." for name in all_class_names]
        tokenized_template_prompts = torch.cat([clip.tokenize(p) for p in template_prompts]).to(self.device) # [all_C, 77]
        with torch.no_grad():
            embedded_template_prompts = clip_model.token_embedding(tokenized_template_prompts).type(self.dtype) # [all_C, 77, D]

        # register embeddings of prefices and suffices
        self.register_buffer("token_prefix", embedded_template_prompts[:, :1, :]) # embedding of SOT (start-of-text token): [all_C, 1, D]
        self.register_buffer("token_suffix", embedded_template_prompts[:, 1+n_ctx:, :]) # embedding of suffix (everything after context + EOT + padding): [all_C, suffix_len, D]
        self.tokenized_template_prompts = tokenized_template_prompts # will be useful later


        # hard-template initialization: take the first template for each expert's group.
        # Split template at class placeholder, tokenize left/right parts separately,
        # concatenate embeddings (excluding class tokens), truncate to n_ctx or pad with noise.
        self.init_from_hard_templates(hard_groups)

        # text encoder
        self.text_encoder = TextEncoder(clip_model, self.token_prefix, self.token_suffix,
                                        self.tokenized_template_prompts)

        # router
        self.router = Router(input_dim=self.clip_model.visual.output_dim, G=self.expert_count, hidden_dim=router_hidden)

        # hard-feature supervision buffers - USE ALL CLASSES for text supervision
        self.register_buffer("hard_group_avg", torch.empty(self.expert_count, text_embedding_dimension))                 # [G, D]
        self.register_buffer("hard_group_class", torch.empty(self.expert_count, self.total_class_count, text_embedding_dimension))  # [G, all_C, D]
        
        # hard-feature supervision buffers for ACTIVE CLASSES ONLY - for router supervision
        self.register_buffer("hard_group_avg_active", torch.empty(self.expert_count, text_embedding_dimension))          # [G, D]

        # reuse CLIP scale
        self.logit_scale = clip_model.logit_scale

    def set_active_classes(self, active_class_indices):
        """Update which classes are active for classification"""
        self.active_class_indices = active_class_indices
        self.active_class_count = len(active_class_indices)
        self.active_class_names = [self.all_class_names[i] for i in active_class_indices]

    @torch.no_grad()
    def set_hard_features(self, hard_group_avg_all, hard_group_class_all):
        """Set hard features for ALL classes (used for text-level supervision)"""
        self.hard_group_avg.copy_(l2norm(hard_group_avg_all))
        self.hard_group_class.copy_(l2norm(hard_group_class_all))
        
        # Also compute hard features for active classes only (for router supervision)
        hard_group_avg_active = []
        for g in range(self.expert_count):
            # Average over active classes only
            active_features = hard_group_class_all[g][self.active_class_indices] # [active_C, D]
            hard_group_avg_active.append(active_features.mean(0, keepdim=True))
        hard_group_avg_active = l2norm(torch.cat(hard_group_avg_active, dim=0))
        self.hard_group_avg_active.copy_(hard_group_avg_active)

    @torch.no_grad()
    def init_from_hard_templates(self, hard_groups):
        """
        Initializes the context parameters for each expert group using the first template in each hard prompt group.
        Args:
            hard_groups (list): List of hard prompt groups, each containing template strings with '{}' as a placeholder.
        Modifies:
            self.ctxs: Sets the context parameters for each group based on tokenized template embeddings.
        """
        token_embedding = self.clip_model.token_embedding
        n_ctx = self.ctxs.shape[1]
        D = self.ctxs.shape[2]

        
        def extract_non_pad_embeddings(text: str):
            """Returns the valid token embeddings excluding the start-of-text (SOT) and end-of-text (EOT) tokens.
                Args:
                    text (str): Prompt template string (without class name).
                Returns:
                    torch.Tensor: Embedded token sequence of shape [L, D], where L is the number of valid tokens
                                  (excluding SOT and EOT) and D is the embedding dimension.
                Notes:
                    - Uses CLIP's tokenizer and token embedding.
                    - Ignores padding tokens.
                    - Returns an empty tensor if the template is empty.
            """
            tokenized_text = clip.tokenize([text]).to(self.device)[0]
            embedded_text  = token_embedding(tokenized_text.unsqueeze(0)).type(self.dtype)[0]
            # valid span = (after SOT) .. (before EOT)
            non_zero_embeddings = (tokenized_text != 0).nonzero(as_tuple=False).flatten() # take indices of non zero tokens (Token ID 0 is padding token)
            if len(non_zero_embeddings) == 0:
                return embedded_text[:0]
            length = int(non_zero_embeddings[-1].item()) + 1 # includes EOT
            return embedded_text[1:length-1] # drop SOT and EOT

        for g, group in enumerate(hard_groups):
            template = group[0]
            if "{}" not in template:
                raise ValueError(f"Template must contain '{{}}': {template}")

            left, right = template.split("{}", 1)
            left_emb  = extract_non_pad_embeddings(left)      # tokens before class
            right_emb = extract_non_pad_embeddings(right)     # tokens after class
            cat = torch.cat([left_emb, right_emb], dim=0)                   # [L',D], no class tokens

            # if ctx is longer than n_ctx -> truncate
            # if ctx is shorter -> pad with small random noise
            if cat.shape[0] >= n_ctx:
                ctx = cat[:n_ctx]
            else:
                pad = torch.randn(n_ctx - cat.shape[0], D, device=self.device, dtype=self.dtype) * 0.01
                ctx = torch.cat([cat, pad], dim=0)

            self.ctxs[g] = ctx

    def encode_all_experts(self):
        """Encodes prompt contexts for all expert groups using the text encoder.
        Returns:
            torch.Tensor: Text features for all experts and all classes, shape [G, all_C, D],
                        where G is the number of expert groups, all_C is the total number of classes,
                        and D is the text embedding dimension.
        """
        features = [self.text_encoder(self.ctxs[g]) for g in range(self.expert_count)]
        return torch.stack(features, dim=0)

    def forward(self, images, return_aux=True):
        """
        Forward pass for PromptExperts.
        Args:
            images (torch.Tensor): Batch of input images, shape [B, ...].
            return_aux (bool, optional): If True, returns auxiliary outputs (router/text regularizers). If False, returns only logits.
        Returns:
            logits (torch.Tensor): Classification logits for active classes, shape [B, active_C].
            aux (dict, optional): Auxiliary outputs including:
                - "gate_logits": Router logits, shape [B, G].
                - "gate_prob": Router probabilities, shape [B, G].
                - "topk_idx": Indices of top-k experts per sample, shape [B, K].
                - "topk_prob": Probabilities of top-k experts per sample, shape [B, K].
                - "loss_router": Router KL regularization loss (scalar).
                - "loss_text": Text-level supervision loss (scalar).
                - "loss_total_reg": Weighted sum of router and text regularization losses (scalar).
        """
        # image features from frozen CLIP
        with torch.no_grad():
            img = self.clip_model.encode_image(images).float() # [B, D] where B is batch size and D is embedding dimension for images. i.e. 512
            img = l2norm(img)

        # router
        gate_logits = self.router(img) # [B, G]
        gate_probs = gate_logits.softmax(dim=-1) # [B, G]

        all_experts_all_classes_text_features = self.encode_all_experts() # [G, all_C, 512]
        active_class_text_features = all_experts_all_classes_text_features[:, self.active_class_indices, :] # [G, active_C, 512]

        # top-k mixture per sample
        batch_size = images.size(0)
        top_k_experts = self.top_k
        topk_probs, topk_idx = gate_probs.topk(top_k_experts, dim=-1) # [B, K], [B, K]
        topk_probs = topk_probs / topk_probs.sum(-1, keepdim=True) # normalize top-k probabilities

        # Get [B, K, active_C, D] text embeddings for top-K experts per sample (ACTIVE CLASSES ONLY)
        txt_embeddings_topk_expert_active_classes = active_class_text_features[topk_idx]  # topk_idx: [B, K] → txt_topk: [B, K, active_C, D]

        # Reshape weights to match: [B, K, 1, 1]
        expert_probability_weights = topk_probs.view(batch_size, top_k_experts, 1, 1)

        # Weighted sum over K experts
        mixed_experts_text_features = (expert_probability_weights * txt_embeddings_topk_expert_active_classes).sum(dim=1) # [B, active_C, D]

        # Normalize across D
        mixed_experts_text_features = l2norm(mixed_experts_text_features, dim=-1)

        # logits for ACTIVE CLASSES ONLY
        logit_scale = self.logit_scale.exp()
        logits = logit_scale * torch.einsum("bd,bcd->bc", img, mixed_experts_text_features) # [B, active_C] cosine similarity between image and text features for each active class

        if not return_aux:
            return logits

        # regularizers: router KL to hard targets, and text-level supervision
        aux = {
            "gate_logits": gate_logits,
            "gate_prob": gate_probs,
            "topk_idx": topk_idx,
            "topk_prob": topk_probs,
        }

        # router KL - use ACTIVE CLASSES ONLY for router supervision
        with torch.no_grad():
            w_hard = (img @ l2norm(self.hard_group_avg_active).T).softmax(dim=-1) # [B, G]
        eps = 1e-8
        loss_router = -(w_hard * gate_probs.clamp_min(eps).log()).sum(dim=-1).mean() # KL divergence in disguise

        # text-level supervision - use ALL CLASSES for text supervision
        txt_soft_all = all_experts_all_classes_text_features # [G, all_C, 512]
        with torch.no_grad():
            hard_norm = l2norm(self.hard_group_class) # [G, all_C, D]
        losses_text = []
        for g in range(self.expert_count):
            logits_g = (txt_soft_all[g] @ hard_norm[g].T) / self.tau # [all_C, all_C]
            target = torch.arange(self.total_class_count, device=images.device)
            losses_text.append(F.cross_entropy(logits_g, target))
        loss_text = torch.stack(losses_text).mean()

        aux["loss_router"] = loss_router
        aux["loss_text"] = loss_text
        aux["loss_total_reg"] = self.lambda_router * loss_router + self.lambda_text * loss_text
        return logits, aux

#### Custom Clip

In [ ]:
class MoCoOpCLIP(nn.Module):
    """MoCoOpCLIP: Mixture of Prompt Learning CLIP Wrapper."""
    def __init__(
            self, clip_model,
            all_class_names,
            active_class_indices=None,
            hard_groups=None,
            n_ctx=16,
            top_k=2,
            tau=0.07,
            lambda_router=1.0,
            lambda_text=5.0,
            router_hidden=0
            ):
        """
        Initializes the MoCoOpCLIP wrapper.
        Args:
            clip_model (nn.Module): Pretrained CLIP model (frozen).
            all_class_names (list): List of all class names.
            active_class_indices (list, optional): Indices of active classes for classification.
            hard_groups (list): List of hard prompt groups.
            n_ctx (int): Number of context tokens.
            top_k (int): Number of top experts to mix.
            tau (float): Temperature for text-level supervision.
            lambda_router (float): Router regularization weight.
            lambda_text (float): Text-level supervision weight.
            router_hidden (int): Hidden dimension for router MLP.
        """
        print(f"MoCoOp instance created | n_ctx={n_ctx}, lambda_text={lambda_text}, router_hidden={router_hidden}")
        super().__init__()
        # freeze CLIP
        for p in clip_model.parameters():
            p.requires_grad = False
        self.clip = clip_model

        # prompt experts
        self.prompt = PromptExperts(
            clip_model=self.clip,
            all_class_names=all_class_names,
            active_class_indices=active_class_indices,
            n_ctx=n_ctx,
            hard_groups=hard_groups,
            top_k=top_k,
            tau=tau,
            lambda_router=lambda_router,
            lambda_text=lambda_text,
            router_hidden=router_hidden
        )

    def set_active_classes(self, active_class_indices):
        """
        Update which classes are active for classification.
        Args:
            active_class_indices (list): Indices of classes to be set as active for classification.
        """
        self.prompt.set_active_classes(active_class_indices)

    @torch.no_grad()
    def prime_hard(self, hard_group_avg, hard_group_class):
        """
        Prime the model with hard features for supervision.
        Args:
            hard_group_avg (torch.Tensor): Hard feature averages for all classes.
            hard_group_class (torch.Tensor): Hard features for all classes.
        """
        self.prompt.set_hard_features(hard_group_avg, hard_group_class)

    def forward(self, images, targets=None):
        """
        Forward pass through the model.
        Args:
            images (torch.Tensor): Batch of input images.
            targets (torch.Tensor, optional): Ground truth labels. If None, only logits and aux are returned.
        Returns:
            logits (torch.Tensor): Classification logits for active classes.
            aux (dict): Auxiliary outputs including losses and router/text regularizers.
        """
        logits, aux = self.prompt(images, return_aux=True)
        if targets is None:
            return logits, aux
        loss_cls = F.cross_entropy(logits, targets)
        loss = loss_cls + aux.get("loss_total_reg", 0.0)
        aux["loss_cls"] = loss_cls
        aux["loss"] = loss
        return logits, aux

    def trainable_parameters(self):
        """
        Returns a list of trainable parameters (prompt contexts and router).
        Returns:
            list: Trainable parameters of the model.
        """
        return [p for p in self.prompt.parameters() if p.requires_grad]

    def save(self, path):
        """
        Save the model state to a file.
        Args:
            path (str): Path to save the model state dictionary.
        """
        torch.save({"prompt_state": self.prompt.state_dict()}, path)

    def load(self, path, strict=True, map_location="cpu"):
        """
        Load the model state from a file.
        Args:
            path (str): Path to load the model state dictionary from.
            strict (bool, optional): Whether to strictly enforce that the keys match.
            map_location (str or torch.device, optional): Device to map the loaded state.
        """
        ckpt = torch.load(path, map_location=map_location)
        self.prompt.load_state_dict(ckpt["prompt_state"], strict=strict)

Training and eval


In [ ]:
# Remap base classes
train_base_remapped = create_remapped_dataset(train_set, base_classes)
val_base_remapped   = create_remapped_dataset(val_set, base_classes)
test_base_remapped  = create_remapped_dataset(test_set, base_classes)

train_base_dataset = RemappedDataset(train_base_remapped)
val_base_dataset   = RemappedDataset(val_base_remapped)
test_base_dataset  = RemappedDataset(test_base_remapped)

batch_size = 32
train_loader = torch.utils.data.DataLoader(train_base_dataset, batch_size=batch_size, shuffle=True,  num_workers=4)
val_loader   = torch.utils.data.DataLoader(val_base_dataset,   batch_size=batch_size, shuffle=False, num_workers=4)
test_loader  = torch.utils.data.DataLoader(test_base_dataset,  batch_size=batch_size, shuffle=False, num_workers=4)

In [ ]:
# Build MoCoOp with ALL classes but only base classes active for training
all_class_names = CLASS_NAMES
mocoop = MoCoOpCLIP(model, all_class_names=all_class_names, active_class_indices=base_classes, hard_groups=HARD_GROUPS,
                    n_ctx=4,
                    lambda_text=5.0,
                    router_hidden=0
                    ).to(device)

# Prime hard features with ALL classes (for text-level supervision)
with torch.no_grad():
    hard_avg_all, hard_cls_all = build_hard_features(model, all_class_names, HARD_GROUPS)
mocoop.prime_hard(hard_avg_all, hard_cls_all)

# Optimizer over trainable parameters only: ctxs + router
params = [p for p in mocoop.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.002, momentum=0.9)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200, eta_min=1e-4)

num_epochs = 25
best_val = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_acc': []}

In [ ]:
def evaluate_mocoop(model, loader, device):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits, _ = model(images)                # targets=None -> returns logits, aux
            pred = logits.argmax(dim=1)
            correct += (pred == labels).sum().item()
            total   += labels.numel()
    return correct / max(total, 1)

print(f"Trainable params: {sum(p.numel() for p in params):,}")

mocoop.train()

In [ ]:
for epoch in range(num_epochs):
    mocoop.train()
    running_loss = 0.0
    correct = total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        images = images.to(torch.float32) # Explicitly cast images to float32

        optimizer.zero_grad()
        logits, aux = mocoop(images, labels)        # returns logits and aux with aux['loss']
        loss = aux['loss']                           # CE + router/text regularizers (already summed inside)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total   += images.size(0)

    scheduler.step()
    train_loss = running_loss / total
    train_acc  = correct / total
    val_acc    = evaluate_mocoop(mocoop, val_loader, device)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    if val_acc > best_val:
        best_val = val_acc
        # Save only prompt state (your class exposes .save/.load)
        mocoop.save('best_mocoop_prompt.pth')
    print(f"Epoch {epoch+1}/{num_epochs} | loss {train_loss:.4f} | acc {train_acc:.4f} | val {val_acc:.4f}")

In [ ]:
print(f"Best val: {best_val:.4f}")

# Load best prompt and test on base
mocoop.load('best_mocoop_prompt.pth', strict=False, map_location=device)   # loads into mocoop.prompt
base_test_acc = evaluate_mocoop(mocoop, test_loader, device)
print(f"Base test acc: {base_test_acc:.4f}")

In [ ]:
# ===== Novel class evaluation =====
# Switch to novel classes (reuse same model, just change active classes)
mocoop.set_active_classes(novel_classes)

# Prime hard features again - still use ALL classes for text supervision, but router uses novel classes
with torch.no_grad():
    hard_avg_all, hard_cls_all = build_hard_features(model, all_class_names, HARD_GROUPS)
mocoop.prime_hard(hard_avg_all, hard_cls_all)

# Novel dataset and loader
test_novel_remapped = create_remapped_dataset(test_set, novel_classes)
test_novel_dataset  = RemappedDataset(test_novel_remapped)
test_novel_loader   = torch.utils.data.DataLoader(test_novel_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

novel_test_acc = evaluate_mocoop(mocoop, test_novel_loader, device)
print(f"Novel test acc: {novel_test_acc:.4f}")
print(f"Harmonic mean: {(2 / (1/base_test_acc + 1/novel_test_acc)):.4f}")